# RGB Object Detection Baseline with YOLO — Low-Memory Training Run

**Author:** Maritza Alban  
**Dataset:** *Seeing Through Fog*  
**Project:** Autonomous Systems — Perception  
**Stage:** Stage 2 — RGB Object Detection Baseline

> **Purpose of this version:** safely restart YOLO training after the previous Jupyter kernel crash. The dataset preparation has already been completed and the YOLO dataset is stored locally. This notebook reuses those files; it does **not** repeat annotation conversion, image copying, or dataset splitting.

The previous run completed the first training epoch (1,128/1,128 batches) but the kernel crashed before a checkpoint (`best.pt` or `last.pt`) was found. Since no checkpoint is available, this run starts again from the pretrained `yolov8n.pt` weights.


## 1. Background and Motivation

The previous RGB analysis characterized image-level brightness, contrast, and sharpness under different environmental conditions. Those measurements describe visual quality, but they do not directly measure whether an object detector can identify road users and obstacles.

This stage establishes a supervised object-detection baseline using the left RGB camera images from *Seeing Through Fog*. The baseline will provide a reference for later robustness experiments and multimodal perception work.


## 2. Objectives and Scope

### General objective

Develop and evaluate an RGB object-detection baseline using YOLOv8n and the prepared *Seeing Through Fog* annotations.

### Specific objectives

1. Reuse the prepared train, validation, and test subsets.
2. Fine-tune a pretrained YOLOv8 Nano detector on six selected object classes.
3. Save model checkpoints and training diagnostics.
4. Establish a baseline for later environmental-condition and multimodal comparisons.

### Scope and limitations

- This is an RGB-only 2D detection baseline; it does not yet perform LiDAR fusion or 3D detection.
- The selected classes are `PassengerCar`, `Pedestrian`, `LargeVehicle`, `RidableVehicle`, `Obstacle`, and `Vehicle`.
- The prepared split was randomly generated at image level (seed 42). Because adjacent driving frames may be correlated, sequence-level leakage is a limitation to consider when interpreting test performance.
- Images without selected-class annotation files were not included as negative/background samples.


## 3. Prepared Dataset and Ground-Truth

The KITTI-like camera annotations were converted to YOLO normalized bounding boxes. Bounding boxes extending beyond image boundaries were clipped to the 1920 × 1024 image dimensions; boxes with zero or negative area after clipping were removed.

The prepared dataset was split into training, validation, and test subsets, and a `data.yaml` configuration file was created. This notebook only checks that those outputs exist and are readable. It deliberately avoids regenerating labels or copying the image dataset.


### 3.1. Restore the project paths and verify the prepared dataset

Run this cell first after selecting the `Python (Autonomous Vehicle)` kernel. It defines all paths needed by this notebook and checks the existing dataset structure.


In [1]:
# ============================================================
# 3.1. Restore project paths and verify the prepared dataset
# ============================================================

from pathlib import Path
import yaml

PROJECT_DIR = Path("/home/maritza/projects/autonomous_vehicle")
YOLO_DATASET_DIR = PROJECT_DIR / "dataset" / "yolo_dataset"
DATA_YAML = YOLO_DATASET_DIR / "data.yaml"

RGB_DIR = (
    PROJECT_DIR
    / "dataset"
    / "seeing_through_fog"
    / "extracted"
    / "cam_stereo_left_lut"
    / "cam_stereo_left_lut"
)

OUTPUT_DIR = PROJECT_DIR / "outputs" / "yolo_baseline"

# Confirm that the prepared dataset is present.
required_paths = [
    DATA_YAML,
    YOLO_DATASET_DIR / "images" / "train",
    YOLO_DATASET_DIR / "images" / "val",
    YOLO_DATASET_DIR / "images" / "test",
    YOLO_DATASET_DIR / "labels" / "train",
    YOLO_DATASET_DIR / "labels" / "val",
    YOLO_DATASET_DIR / "labels" / "test",
]

missing_paths = [path for path in required_paths if not path.exists()]

if missing_paths:
    print("Missing required dataset paths:")
    for path in missing_paths:
        print(f" - {path}")
    raise FileNotFoundError(
        "The prepared YOLO dataset is incomplete. Stop here; do not start training."
    )

with open(DATA_YAML, "r") as file:
    dataset_config = yaml.safe_load(file)

print("Prepared dataset found.")
print(f"Dataset YAML: {DATA_YAML}")
print(f"RGB source directory: {RGB_DIR}")
print("\nDataset configuration:")
print(dataset_config)

for split in ["train", "val", "test"]:
    image_dir = YOLO_DATASET_DIR / "images" / split
    label_dir = YOLO_DATASET_DIR / "labels" / split
    image_count = len(list(image_dir.glob("*.png")))
    label_count = len(list(label_dir.glob("*.txt")))
    print(
        f"{split:>5}: {image_count:,} images | "
        f"{label_count:,} label files"
    )


Prepared dataset found.
Dataset YAML: /home/maritza/projects/autonomous_vehicle/dataset/yolo_dataset/data.yaml
RGB source directory: /home/maritza/projects/autonomous_vehicle/dataset/seeing_through_fog/extracted/cam_stereo_left_lut/cam_stereo_left_lut

Dataset configuration:
{'path': '/home/maritza/projects/autonomous_vehicle/dataset/yolo_dataset', 'train': 'images/train', 'val': 'images/val', 'test': 'images/test', 'nc': 6, 'names': ['PassengerCar', 'Pedestrian', 'LargeVehicle', 'RidableVehicle', 'Obstacle', 'Vehicle']}
train: 9,018 images | 9,018 label files
  val: 1,932 images | 1,932 label files
 test: 1,933 images | 1,933 label files


### 3.2. Confirm the compute environment

We will use the NVIDIA GPU for model training. The previous run showed that GPU memory usage was modest, while system RAM and swap were close to exhaustion. The main adjustment in this run is therefore to reduce CPU-side data-loading pressure.


In [2]:
# ============================================================
# 3.2. Check PyTorch and CUDA
# ============================================================

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Check the selected Python kernel and PyTorch installation."
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB"
)


PyTorch version: 2.15.0+cu132
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
GPU memory: 7.96 GB


## 4. YOLO Baseline Implementation

### 4.1. Load the pretrained YOLOv8 Nano model

We use the Nano variant (`yolov8n`) to keep the baseline computationally lightweight. Its pretrained weights provide an initialization learned from a general object-detection dataset; fine-tuning adapts the detector to the six classes and image characteristics in *Seeing Through Fog*.


In [3]:
# ============================================================
# 4.1. Load the pretrained YOLOv8 Nano model
# ============================================================

from ultralytics import YOLO

MODEL_WEIGHTS = PROJECT_DIR / "yolov8n.pt"

if not MODEL_WEIGHTS.exists():
    raise FileNotFoundError(
        f"Pretrained weights not found: {MODEL_WEIGHTS}"
    )

model = YOLO(str(MODEL_WEIGHTS))

print(f"Model loaded: {MODEL_WEIGHTS}")
print("Model task:", model.task)


Model loaded: /home/maritza/projects/autonomous_vehicle/yolov8n.pt
Model task: detect


### 4.2. Configure a lower-memory training run

The settings below are intentionally conservative for the current WSL environment:

- `batch=4`: halves the number of images processed per batch compared with the previous attempt (`batch=8`).
- `workers=0`: disables additional DataLoader worker processes, reducing CPU RAM overhead and avoiding multiprocessing complications in notebook/WSL sessions.
- `cache=False`: prevents caching the dataset in RAM.
- `imgsz=640`: retains the previous input resolution so the baseline remains comparable.
- `device=0`: uses the RTX 5060 GPU.
- A new run name (`yolov8n_rgb_lowmem`) keeps this attempt separate from the previous incomplete run.

Because no checkpoint from the previous run was found, training starts from the pretrained weights rather than resuming.


In [4]:
# ============================================================
# 4.2. Configure the lower-memory training run
# ============================================================

TRAINING_CONFIG = {
    "data": str(DATA_YAML),
    "epochs": 50,
    "imgsz": 640,
    "batch": 4,
    "device": 0,
    "workers": 0,
    "cache": False,
    "patience": 10,
    "seed": 42,
    "project": str(OUTPUT_DIR),
    "name": "yolov8n_rgb_lowmem",
    "exist_ok": True,
    "plots": True,
    "save": True,
    "verbose": True
}

print("YOLO training configuration:\n")
for parameter, value in TRAINING_CONFIG.items():
    print(f"{parameter}: {value}")


YOLO training configuration:

data: /home/maritza/projects/autonomous_vehicle/dataset/yolo_dataset/data.yaml
epochs: 50
imgsz: 640
batch: 4
device: 0
workers: 0
cache: False
patience: 10
seed: 42
project: /home/maritza/projects/autonomous_vehicle/outputs/yolo_baseline
name: yolov8n_rgb_lowmem
exist_ok: True
plots: True
save: True
verbose: True


### 4.3. Train the RGB baseline

Run this cell only after the dataset and CUDA checks have completed successfully. Training may take some time. Avoid opening additional notebooks or launching other memory-intensive applications while it runs.

YOLO periodically saves checkpoints in the run directory. If the kernel stops again, inspect the `weights` subdirectory before deciding whether to resume or restart.


In [ ]:
# ============================================================
# 4.3. Train the YOLO RGB baseline
# ============================================================

training_results = model.train(**TRAINING_CONFIG)


Ultralytics 8.4.204 🚀 Python-3.12.3 torch-2.15.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/maritza/projects/autonomous_vehicle/dataset/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/home/maritza/projects/autonomous_vehicle/yolov8n.pt, mo

### 4.4. Verify the training outputs

After training finishes, this cell checks whether YOLO saved the expected checkpoints and run artifacts.


In [ ]:
# ============================================================
# 4.4. Verify saved training artifacts
# ============================================================

RUN_DIR = OUTPUT_DIR / "yolov8n_rgb_lowmem"
WEIGHTS_DIR = RUN_DIR / "weights"

print("Run directory exists:", RUN_DIR.exists())

if WEIGHTS_DIR.exists():
    for checkpoint_name in ["best.pt", "last.pt"]:
        checkpoint_path = WEIGHTS_DIR / checkpoint_name
        if checkpoint_path.exists():
            size_mb = checkpoint_path.stat().st_size / (1024 ** 2)
            print(f"Found {checkpoint_name}: {size_mb:.1f} MB")
        else:
            print(f"{checkpoint_name}: not found")
else:
    print("Weights directory has not been created.")


In [2]:
from pathlib import Path

RUN_DIR = (
    Path("/home/maritza/projects/autonomous_vehicle")
    / "outputs/yolo_baseline/yolov8n_rgb_lowmem"
)

LAST_CHECKPOINT = RUN_DIR / "weights" / "last.pt"
BEST_CHECKPOINT = RUN_DIR / "weights" / "best.pt"

for checkpoint in [LAST_CHECKPOINT, BEST_CHECKPOINT]:
    print(
        checkpoint.name,
        "exists:", checkpoint.exists(),
        "size:", f"{checkpoint.stat().st_size / 1024**2:.2f} MB"
    )

last.pt exists: True size: 30.30 MB
best.pt exists: True size: 30.30 MB


In [3]:
from ultralytics import YOLO

LAST_CHECKPOINT = (
    "/home/maritza/projects/autonomous_vehicle/"
    "outputs/yolo_baseline/yolov8n_rgb_lowmem/weights/last.pt"
)

checkpoint_model = YOLO(LAST_CHECKPOINT)

print("Checkpoint loaded successfully.")
print("Model task:", checkpoint_model.task)
print("Checkpoint:", LAST_CHECKPOINT)

Checkpoint loaded successfully.
Model task: detect
Checkpoint: /home/maritza/projects/autonomous_vehicle/outputs/yolo_baseline/yolov8n_rgb_lowmem/weights/last.pt


In [4]:
model = YOLO(LAST_CHECKPOINT)
results = model.train(resume=True)

Ultralytics 8.4.204 🚀 Python-3.12.3 torch-2.15.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/maritza/projects/autonomous_vehicle/dataset/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/home/maritza/projects/autonomous_vehicle/outputs/yolo_

In [5]:
# ============================================================
# Save a backup of the trained model
# ============================================================

import shutil

BACKUP_DIR = PROJECT_DIR / "models" / "yolov8n_rgb"
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(
    BEST_CHECKPOINT,
    BACKUP_DIR / "best.pt"
)

shutil.copy2(
    BEST_CHECKPOINT.parent / "last.pt",
    BACKUP_DIR / "last.pt"
)

print("Model checkpoints backed up successfully.")
print("Backup directory:", BACKUP_DIR)

Model checkpoints backed up successfully.
Backup directory: /home/maritza/projects/autonomous_vehicle/models/yolov8n_rgb


In [6]:
# ============================================================
# 5.1. Load the trained YOLO model
# ============================================================

from pathlib import Path
from ultralytics import YOLO

PROJECT_DIR = Path("/home/maritza/projects/autonomous_vehicle")

BEST_CHECKPOINT = (
    PROJECT_DIR
    / "outputs"
    / "yolo_baseline"
    / "yolov8n_rgb_lowmem"
    / "weights"
    / "best.pt"
)

DATA_YAML = (
    PROJECT_DIR
    / "dataset"
    / "yolo_dataset"
    / "data.yaml"
)

# Verify the required files
assert BEST_CHECKPOINT.exists(), "Trained model not found."
assert DATA_YAML.exists(), "Dataset configuration not found."

# Load the best model
model = YOLO(str(BEST_CHECKPOINT))

print("Model loaded successfully.")
print("Checkpoint:", BEST_CHECKPOINT)
print("Dataset configuration:", DATA_YAML)
print("Model task:", model.task)

Model loaded successfully.
Checkpoint: /home/maritza/projects/autonomous_vehicle/outputs/yolo_baseline/yolov8n_rgb_lowmem/weights/best.pt
Dataset configuration: /home/maritza/projects/autonomous_vehicle/dataset/yolo_dataset/data.yaml
Model task: detect


# Model with favorable conditions

In [9]:
# ============================================================
# 1. Environment and Project Configuration
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import yaml
import shutil

PROJECT_DIR = Path("/home/maritza/projects/autonomous_vehicle")

ORIGINAL_DATASET_DIR = PROJECT_DIR / "dataset" / "yolo_dataset"

CLEAR_DATASET_DIR = (
    PROJECT_DIR / "dataset" / "yolo_dataset_clear_weather"
)

ENVIRONMENT_CSV = (
    PROJECT_DIR
    / "outputs"
    / "stage1_rgb_baseline"
    / "rgb_image_quality_with_environmental_labels.csv"
)

OUTPUT_DIR = PROJECT_DIR / "outputs" / "yolo_baseline"

# Load original dataset configuration
with open(ORIGINAL_DATASET_DIR / "data.yaml", "r") as file:
    original_config = yaml.safe_load(file)

CLASS_NAMES = original_config["names"]

print("Project:", PROJECT_DIR)
print("Original dataset:", ORIGINAL_DATASET_DIR)
print("Clear-weather dataset:", CLEAR_DATASET_DIR)
print("Classes:", CLASS_NAMES)

# Load environmental metadata
environment_df = pd.read_csv(ENVIRONMENT_CSV)

print("\nEnvironmental metadata:")
print("Images:", len(environment_df))
print("Columns:", environment_df.columns.tolist())

# ============================================================
# 2. Identify Clear-Weather Images
# ============================================================

favorable_mask = (
    (environment_df["fog_condition"] == "none") &
    (environment_df["precipitation_condition"] == "none") &
    (environment_df["daytime_condition"] == "day") &
    (environment_df["road_condition"] == "dry")
)

favorable_df = environment_df.loc[favorable_mask].copy()

# Identify the image filename column
filename_candidates = [
    "image_key",
    "image_filename",
    "filename",
    "file_name",
    "image_name",
    "rgb_filename",
    "image_path",
    "path"
]

filename_column = next(
    (
        col for col in filename_candidates
        if col in favorable_df.columns
    ),
    None
)

if filename_column is None:
    raise ValueError(
        "Could not identify the image filename column. "
        "Inspect environment_df.columns."
    )

favorable_df["image_key"] = (
    favorable_df[filename_column]
    .apply(lambda x: Path(str(x)).name)
)

print("Favorable images:", len(favorable_df))
print(
    f"Percentage of environmental dataset: "
    f"{len(favorable_df) / len(environment_df) * 100:.2f}%"
)

display(favorable_df.head())



Project: /home/maritza/projects/autonomous_vehicle
Original dataset: /home/maritza/projects/autonomous_vehicle/dataset/yolo_dataset
Clear-weather dataset: /home/maritza/projects/autonomous_vehicle/dataset/yolo_dataset_clear_weather
Classes: ['PassengerCar', 'Pedestrian', 'LargeVehicle', 'RidableVehicle', 'Obstacle', 'Vehicle']

Environmental metadata:
Images: 12997
Columns: ['image_name', 'image_path', 'width', 'height', 'channels', 'dtype', 'brightness_mean', 'contrast_std', 'sharpness_laplacian_var', 'pixel_min', 'pixel_max', 'fog_condition', 'precipitation_condition', 'daytime_condition', 'twilight', 'road_condition']
Favorable images: 1594
Percentage of environmental dataset: 12.26%


,image_name,image_path,width,height,channels,dtype,brightness_mean,contrast_std,sharpness_laplacian_var,pixel_min,pixel_max,fog_condition,precipitation_condition,daytime_condition,twilight,road_condition,image_key
318,2018-02-04_10-00-03_00200.png,/home/maritza/projects/autonomous_vehicle/data...,1920,1024,3,uint8,99.261907,49.273588,132.510640,0,241,none,none,day,False,dry,2018-02-04_10-00-03_00200.png
319,2018-02-04_10-00-43_00000.png,/home/maritza/projects/autonomous_vehicle/data...,1920,1024,3,uint8,99.047174,46.358260,131.465729,0,246,none,none,day,False,dry,2018-02-04_10-00-43_00000.png
320,2018-02-04_10-00-43_00100.png,/home/maritza/projects/autonomous_vehicle/data...,1920,1024,3,uint8,102.079404,54.201456,122.354807,0,254,none,none,day,False,dry,2018-02-04_10-00-43_00100.png
321,2018-02-04_10-00-43_00200.png,/home/maritza/projects/autonomous_vehicle/data...,1920,1024,3,uint8,97.527682,51.367404,126.208295,0,255,none,none,day,False,dry,2018-02-04_10-00-43_00200.png
322,2018-02-04_10-00-43_00300.png,/home/maritza/projects/autonomous_vehicle/data...,1920,1024,3,uint8,102.663683,51.512734,173.114386,0,248,none,none,day,False,dry,2018-02-04_10-00-43_00300.png


In [ ]:
# ============================================================
# 3. Build the Clear-Weather Training Dataset
# ============================================================

favorable_names = set(favorable_df["image_key"])

split_sources = {
    "train": ORIGINAL_DATASET_DIR,
    "val": ORIGINAL_DATASET_DIR
}

split_counts = []

# Create a separate dataset directory
for split in ["train", "val"]:

    source_images = (
        ORIGINAL_DATASET_DIR / "images" / split
    )

    source_labels = (
        ORIGINAL_DATASET_DIR / "labels" / split
    )

    destination_images = (
        CLEAR_DATASET_DIR / "images" / split
    )

    destination_labels = (
        CLEAR_DATASET_DIR / "labels" / split
    )

    destination_images.mkdir(
        parents=True,
        exist_ok=True
    )

    destination_labels.mkdir(
        parents=True,
        exist_ok=True
    )

    available_images = {
        p.name for p in source_images.glob("*.png")
    }

    selected_images = sorted(
        available_images & favorable_names
    )

    copied_images = 0
    copied_objects = 0

    for image_name in selected_images:

        image_path = source_images / image_name

        label_path = (
            source_labels /
            f"{Path(image_name).stem}.txt"
        )

        if not label_path.exists():
            continue

        # Copy image and its corresponding annotation
        shutil.copy2(
            image_path,
            destination_images / image_name
        )

        shutil.copy2(
            label_path,
            destination_labels / label_path.name
        )

        with open(label_path, "r") as file:
            copied_objects += sum(
                1 for line in file if line.strip()
            )

        copied_images += 1

    split_counts.append({
        "Split": split,
        "Images": copied_images,
        "Objects": copied_objects
    })

clear_split_summary = pd.DataFrame(split_counts)

display(clear_split_summary)

# ============================================================
# 4. Analyze Class Distribution
# ============================================================

class_distribution = []

for split in ["train", "val"]:

    label_dir = (
        CLEAR_DATASET_DIR / "labels" / split
    )

    class_counts = {
        class_id: 0
        for class_id in range(len(CLASS_NAMES))
    }

    for label_path in label_dir.glob("*.txt"):

        with open(label_path, "r") as file:

            for line in file:

                values = line.strip().split()

                if values:
                    class_id = int(values[0])
                    class_counts[class_id] += 1

    for class_id, count in class_counts.items():

        class_distribution.append({
            "Split": split,
            "Class": CLASS_NAMES[class_id],
            "Objects": count
        })

class_distribution_df = pd.DataFrame(
    class_distribution
)

display(
    class_distribution_df.pivot(
        index="Class",
        columns="Split",
        values="Objects"
    ).fillna(0).astype(int)
)

# ============================================================
# 5. Create Clear-Weather Dataset Configuration
# ============================================================

clear_data_config = {
    "path": str(CLEAR_DATASET_DIR),
    "train": "images/train",
    "val": "images/val",
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES
}

CLEAR_DATA_YAML = CLEAR_DATASET_DIR / "data.yaml"

with open(CLEAR_DATA_YAML, "w") as file:
    yaml.safe_dump(
        clear_data_config,
        file,
        sort_keys=False
    )

print("Dataset configuration saved:")
print(CLEAR_DATA_YAML)

print("\nConfiguration:")
print(clear_data_config)

# ============================================================
# 6. Initialize the Clear-Weather YOLO Model
# ============================================================

from ultralytics import YOLO

PRETRAINED_WEIGHTS = PROJECT_DIR / "yolov8n.pt"

clear_model = YOLO(str(PRETRAINED_WEIGHTS))

print("Independent model initialized.")
print("Architecture: YOLOv8n")
print("Initial weights:", PRETRAINED_WEIGHTS)

,Split,Images,Objects
0,train,1120,13338
1,val,234,3025


Split,train,val
Class,,
LargeVehicle,462,110
Obstacle,366,104
PassengerCar,6289,1383
Pedestrian,5352,1282
RidableVehicle,616,109
Vehicle,253,37


Dataset configuration saved:
/home/maritza/projects/autonomous_vehicle/dataset/yolo_dataset_clear_weather/data.yaml

Configuration:
{'path': '/home/maritza/projects/autonomous_vehicle/dataset/yolo_dataset_clear_weather', 'train': 'images/train', 'val': 'images/val', 'nc': 6, 'names': ['PassengerCar', 'Pedestrian', 'LargeVehicle', 'RidableVehicle', 'Obstacle', 'Vehicle']}


In [ ]:
# ============================================================
# 7. Configure Clear-Weather Training
# ============================================================

CLEAR_TRAINING_CONFIG = {
    "data": str(CLEAR_DATA_YAML),
    "epochs": 50,
    "imgsz": 640,
    "batch": 4,
    "device": 0,
    "workers": 0,
    "cache": False,
    "patience": 10,
    "seed": 42,
    "project": str(OUTPUT_DIR),
    "name": "yolov8n_rgb_clear_weather",
    "exist_ok": False,
    "plots": True,
    "save": True,
    "verbose": True
}

print("Clear-weather training configuration:\n")

for parameter, value in CLEAR_TRAINING_CONFIG.items():
    print(f"{parameter}: {value}")



Clear-weather training configuration:

data: /home/maritza/projects/autonomous_vehicle/dataset/yolo_dataset_clear_weather/data.yaml
epochs: 50
imgsz: 640
batch: 4
device: 0
workers: 0
cache: False
patience: 10
seed: 42
project: /home/maritza/projects/autonomous_vehicle/outputs/yolo_baseline
name: yolov8n_rgb_clear_weather
exist_ok: False
plots: True
save: True
verbose: True


In [15]:
# ============================================================
# 8. Train the Clear-Weather Specialist
# ============================================================

clear_training_results = clear_model.train(
    **CLEAR_TRAINING_CONFIG
)

Ultralytics 8.4.204 🚀 Python-3.12.3 torch-2.15.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/maritza/projects/autonomous_vehicle/dataset/yolo_dataset_clear_weather/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/home/maritza/projects/autonomous_vehicle

In [ ]:
# ============================================================
# 9. Save Clear-Weather Model Checkpoints
# ============================================================

CLEAR_RUN_DIR = (
    OUTPUT_DIR / "yolov8n_rgb_clear_weather"
)

CLEAR_WEIGHTS_DIR = CLEAR_RUN_DIR / "weights"

CLEAR_MODEL_DIR = (
    PROJECT_DIR / "models" / "yolov8n_rgb_clear_weather"
)

CLEAR_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

for checkpoint_name in ["best.pt", "last.pt"]:

    source = CLEAR_WEIGHTS_DIR / checkpoint_name
    destination = CLEAR_MODEL_DIR / checkpoint_name

    if not source.exists():
        raise FileNotFoundError(
            f"Checkpoint not found: {source}"
        )

    shutil.copy2(source, destination)

    print(
        f"Saved {checkpoint_name}: "
        f"{destination.stat().st_size / 1024**2:.2f} MB"
    )

print("\nClear-weather model saved successfully.")

# ============================================================
# Load Both Models for Comparative Evaluation
# ============================================================

from ultralytics import YOLO

GENERAL_MODEL_PATH = (
    PROJECT_DIR / "models" / "yolov8n_rgb" / "best.pt"
)

CLEAR_MODEL_PATH = (
    PROJECT_DIR
    / "models"
    / "yolov8n_rgb_clear_weather"
    / "best.pt"
)

general_model = YOLO(str(GENERAL_MODEL_PATH))
clear_model = YOLO(str(CLEAR_MODEL_PATH))

print("Generalist model loaded:", GENERAL_MODEL_PATH)
print("Clear-weather model loaded:", CLEAR_MODEL_PATH)



Saved best.pt: 6.19 MB
Saved last.pt: 6.19 MB

Clear-weather model saved successfully.
Generalist model loaded: /home/maritza/projects/autonomous_vehicle/models/yolov8n_rgb/best.pt
Clear-weather model loaded: /home/maritza/projects/autonomous_vehicle/models/yolov8n_rgb_clear_weather/best.pt


: 